In [7]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from scipy import stats


def pearsonr_ci(df, col_x, col_y, alpha=0.05):
    '''
        https://zhiyzuo.github.io/Pearson-Correlation-CI-in-Python/
    '''
    
    df_new = df[[col_x, col_y]].dropna()
    n = len(df_new) - 3

    r, p = stats.pearsonr(df_new[col_x], df_new[col_y])
    r_z = np.arctanh(r)
    se = 1 / np.sqrt(n - 3)
    z = stats.norm.ppf(1 - alpha / 2)
    lo_z, hi_z = r_z - z * se, r_z + z * se
    lo, hi = np.tanh((lo_z, hi_z))

    return r, p, lo, hi


def rbf_kernel(X, sigma=1):
    sq_dists = np.sum(X ** 2, axis=1).reshape(-1, 1) + np.sum(X ** 2, axis=1) - 2 * np.dot(X, X.T)
    K = np.exp(-sq_dists / (2 * sigma**2))
    return K

In [13]:
def generate_rct_selection(row):
    return np.random.binomial(1, 0.9)


def generation_obs_selection(row):
    return np.random.binomial(1, 0.9)


# def generation_obs_selection(row):
#     if row["X1"] == -1:
#         return np.random.binomial(1, 0.1)
#     else: 
#         if row["U1"] == -1:
#             return np.random.binomial(1, 0.9)
#         else:
#             return np.random.binomial(1, 0.1)


def generate_rct_treatment(row):
    if row['S'] == 0:
        return np.nan
    else:
        return np.random.binomial(1, 0.5)


def generate_obs_treatment(row):
    if row['S'] == 0:
        return np.nan
    else:
        return np.random.binomial(1, 0.5)


def generate_obs_treatment(row):
    if row['S'] == 0:
        return np.nan
    else:
        if row["X1"] == -1:
            return np.random.binomial(1, 0.1)
        else:
            if row["U1"] == -1:
                return np.random.binomial(1, 0.1)
            else:
                return np.random.binomial(1, 0.9)


def generate_outcome(row):
    if row['S'] == 0:
        return np.nan
    else:
        if row['A'] == 0:
            return np.random.normal(0, 0.1)
        else:
            if row["X1"] == -1:
                return row["X1"] + np.random.normal(0, 0.1)
            else:
                return 1 + row["X1"] + 2 * row["U1"] + np.random.normal(0, 0.1)


def generate_treatment_outcome_selection(df, fn_tr, fn_out, fn_sel):
    df["S"] = df.apply(fn_sel, axis=1) 
    df["A"] = df.apply(fn_tr, axis=1)
    df["Y"] = df.apply(fn_out, axis=1)


def log_om_res(df, covs, model): 
    if model == "DTR":
        om_A0 = DecisionTreeRegressor().fit(df.query("S==1 & A==0")[covs], df.query("S==1 & A==0")["Y"])
        om_A1 = DecisionTreeRegressor().fit(df.query("S==1 & A==1")[covs], df.query("S==1 & A==1")["Y"])

    df["hat_Y0"] = om_A0.predict(df[covs])  # log counter-factual outcome predictions for T=0
    df["hat_Y1"] = om_A1.predict(df[covs])  # log counter-factual outcome predictions for T=1

    df["SE_Y0"] = (df.query("A==0")["Y"] - df.query("A==0")["hat_Y0"]) ** 2  # log instance-wise squared-error of the outcome model for T=0
    df["SE_Y1"] = (df.query("A==1")["Y"] - df.query("A==1")["hat_Y1"]) ** 2  # log instance-wise squared-error of the outcome model for T=1
 

def log_ps_res(df, covs, model): 
    if model == "DTC":
        psm = DecisionTreeClassifier().fit(df.query("S==1")[covs], df.query("S==1")["A"])

    df["hat_P(A=1)"] = psm.predict_proba(df[covs])[:,-1]
    df["BCELoss_A"] = - df["A"] * np.log(df["hat_P(A=1)"]) - (1 - df["A"]) * np.log(1 - df["hat_P(A=1)"]) 
    df["SE_A"] = (df["A"] - df["hat_P(A=1)"]) ** 2 


def log_sm_res(df, covs, model): 
    if model == "DTC":
        sm = DecisionTreeClassifier().fit(df[covs], df["S"])

    df["hat_P(S=1)"] = sm.predict_proba(df[covs])[:,-1]
    df["BCELoss_S"] = - df["S"] * np.log(df["hat_P(S=1)"]) - (1 - df["S"]) * np.log(1 - df["hat_P(S=1)"]) 
    df["SE_S"] = (df["S"] - df["hat_P(S=1)"]) ** 2 


def log_rm_res(df, covs, model): 
    if model == "DTC":
        rm = DecisionTreeClassifier().fit(df[covs], df["R"])

    df["hat_P(R=1)"] = rm.predict_proba(df[covs])[:,-1]
    df["BCELoss_R"] = - df["R"] * np.log(df["hat_P(R=1)"]) - (1 - df["R"]) * np.log(1 - df["hat_P(R=1)"]) 
    df["SE_R"] = (df[ "R"] - df["hat_P(R=1)"]) ** 2
 

def psi0(row):
    if row["R"] == 1 or row["S"] == 0:
        return 0
    else:
        a_ind = row["A"]
        p_r1, p_a1, p_s1 = row["hat_P(R=1)"], row["hat_P(A=1)"], row["hat_P(S=1)"]
        y, mu0, mu1 = row["Y"], row["hat_Y0"], row["hat_Y1"]

        term_1 = mu1 - mu0
        term_2 = a_ind * (y - mu1) / p_a1
        term_3 = (1 - a_ind) * (y - mu0) / (1 - p_a1)

        return (term_1 + term_2 - term_3) / ((1 - p_r1) * p_s1)


def psi1(row):
    if row["R"] == 0 or row["S"] == 0:
        return 0
    else:
        a_ind = row["A"]
        p_r1, p_a1, p_s1 = row["hat_P(R=1)"], row["hat_P(A=1)"], row["hat_P(S=1)"]
        y, mu0, mu1 = row["Y"], row["hat_Y0"], row["hat_Y1"]

        term_1 = mu1 - mu0
        term_2 = a_ind * (y - mu1) / p_a1
        term_3 = (1 - a_ind) * (y - mu0) / (1 - p_a1)

        return (term_1 + term_2 - term_3) / (p_r1 * p_s1)
    

def calc_contrasts(df):
    df["psi0"] = df.apply(psi0, axis=1)
    df["psi1"] = df.apply(psi1, axis=1)
    df["psi"] = df["psi1"] - df["psi0"]

In [14]:
np.random.seed(42)

n_covs = 1
n_unmeasured_covs = 1

n_rct = 1000
n_obs = 5000

covs = [f'X{i+1}' for i in range(n_covs)] 
unmeasured_covs = [f'U{i+1}' for i in range(n_unmeasured_covs)] 

X_rct = np.random.choice([-1, 1], size=(n_rct, n_covs), p=[0.5, 0.5])
U_rct = np.random.choice([-1, 1], size=(n_rct, n_unmeasured_covs), p=[0.5, 0.5])

df_rct = pd.DataFrame({**{cov: X_rct[:,i] for i, cov in enumerate(covs)},
                        **{u_cov: U_rct[:,i] for i, u_cov in enumerate(unmeasured_covs)},
                        **{'R': 1}})

X_obs = np.random.choice([-1, 1], size=(n_obs, n_covs), p=[0.5, 0.5])
U_obs = np.random.choice([-1, 1], size=(n_obs, n_unmeasured_covs), p=[0.5, 0.5])

df_obs = pd.DataFrame({**{cov: X_obs[:,i] for i, cov in enumerate(covs)},
                        **{u_cov: U_obs[:,i] for i, u_cov in enumerate(unmeasured_covs)},
                        **{'R': 0}})

In [15]:
obs_covs = covs + unmeasured_covs
obs_covs = covs

generate_treatment_outcome_selection(df_rct, generate_rct_treatment, generate_outcome, generate_rct_selection)
log_sm_res(df_rct, obs_covs, "DTC")
log_om_res(df_rct, obs_covs, "DTR")
log_ps_res(df_rct, obs_covs, "DTC")

generate_treatment_outcome_selection(df_obs, generate_obs_treatment, generate_outcome, generation_obs_selection)
log_sm_res(df_obs, obs_covs, "DTC")
log_om_res(df_obs, obs_covs, "DTR")
log_ps_res(df_obs, obs_covs, "DTC")

df_merged = pd.concat([df_rct, df_obs]).reset_index(drop=True)
log_rm_res(df_merged, obs_covs, "DTC")

calc_contrasts(df_merged)

K = rbf_kernel(np.atleast_2d(df_merged[obs_covs]))
df_merged['w(X)'] = abs(K @ df_merged['psi'])

In [16]:
print(f"Avg. Bias: {df_merged['psi'].mean():.2f}")
print(f"Bias (X1=-1): {df_merged.query('X1==-1')['psi'].mean():.2f}")
print(f"Bias (X1=1): {df_merged.query('X1==1')['psi'].mean():.2f}")

Avg. Bias: -0.71
Bias (X1=-1): 0.01
Bias (X1=1): -1.45


In [17]:
m, p, lb, ub = pearsonr_ci(df_merged.query("R==0 & S==1"), "w(X)", "SE_Y0")
print(m, p)
m, p, lb, ub = pearsonr_ci(df_merged.query("R==0 & S==1"), "w(X)", "SE_Y1")
print(m, p)
m, p, lb, ub = pearsonr_ci(df_merged.query("R==0 & S==1"), "w(X)", "SE_A")
print(m, p)
m, p, lb, ub = pearsonr_ci(df_merged, "w(X)", "SE_S")
print(m, p)
m, p, lb, ub = pearsonr_ci(df_merged, "w(X)", "SE_R")
print(m, p)

0.0017474123740566635 0.9219173241924854
0.17858439339399493 4.651124933232394e-11
0.3989453883931625 4.0702682342907205e-171
-0.0006147728556415502 0.9620269161145955
0.0031306594442786374 0.8084315680240463


In [ ]:
df_merged